# Week 8-C: Counterfactual Passage Utility Modeling
## Predicting Marginal Promotion Utility $(q, y_0, D_i) \longrightarrow \Delta_i$

Trains and evaluates:
- **U0**: Tabular Baseline (GBDT + Logistic) on engineered features.
- **U1**: Pre-generation Query/Passage Cross-Encoder.
- **U2**: Draft-Conditioned Cross-Encoder $[q, y_0, D_i]$ with 5 multitask heads.
- **U4**: Full Hybrid Fusion Model (Cross-Encoder + MIG + Attribution Features).

### Multitask Prediction Targets:
1. $P(\text{HELP}_i)$: Binary classification ($\Delta F1 > 0.01$).
2. $P(\text{HURT}_i)$: Binary classification ($\Delta F1 < -0.01$).
3. $\widehat{\Delta F1}_i$: Continuous F1 delta regression.
4. $\widehat{\Delta LL}_i$: Dense gold log-likelihood change ($L_i - L_0$).
5. 3-Way label (HELP / NEUTRAL / HURT).


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import torch

REPO_ROOT = Path("/home/gunavenkat/Downloads/CS787-RAG-Project")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.week8.utility_dataset import (
    build_utility_dataset,
    get_non_leaking_feature_columns,
)
from src.week8.utility_models import (
    UtilityModelU0,
    CrossEncoderUtilityModel,
    HybridFusionUtilityModelU4,
    prepare_cross_encoder_inputs,
)

# Load promotion surface and draft diagnostics
split = "internal_val"
prom_path = REPO_ROOT / "results/week8/surfaces" / f"promotion_surface_{split}.jsonl"
draft_path = REPO_ROOT / "results/week8/drafts" / f"draft_diagnostics_{split}.jsonl"

df_prom = pd.read_json(prom_path, lines=True)
df_draft = pd.read_json(draft_path, lines=True) if draft_path.exists() else None

df_util = build_utility_dataset(df_prom, df_draft)
non_leaking_cols = get_non_leaking_feature_columns(df_util)
print(f"Assembled utility dataset with {len(df_util)} candidate rows.")
print(f"Identified {len(non_leaking_cols)} non-leaking feature columns.")


In [ ]:
# Fit and Evaluate Tabular Baseline U0
X = df_util[non_leaking_cols].fillna(0.0).values
y_help = df_util["target_help_binary"].values
y_hurt = df_util["target_hurt_binary"].values
y_delta = df_util["target_continuous_delta"].values
y_3way = df_util["target_3way_label"].values
y_delta_ll = df_util["target_dense_gold_ll"].values if "target_dense_gold_ll" in df_util else None

u0 = UtilityModelU0()
u0.fit(X, y_help, y_hurt, y_delta, y_3way, y_delta_ll=y_delta_ll)
preds_u0 = u0.predict(X)

print("=== U0 Tabular Baseline Predictions ===")
print(f"Mean Pred Prob(HELP): {preds_u0['prob_help'].mean():.4f}")
print(f"Mean Pred Prob(HURT): {preds_u0['prob_hurt'].mean():.4f}")
print(f"Mean Pred Delta F1:   {preds_u0['pred_delta_f1'].mean():.4f}")


In [ ]:
# Evaluate Draft-Conditioned Cross-Encoder U2
# Formats input as: [CLS] Query [SEP] Draft y0 [SEP] Candidate Di [SEP]
device = "cuda" if torch.cuda.is_available() else "cpu"
u2_model = CrossEncoderUtilityModel(freeze_backbone=True).to(device)
print("U2 Draft-Conditioned Cross-Encoder instantiated with 5 multitask heads.")

# Save evaluation summary table
out_table_path = REPO_ROOT / "results/week8/tables/w8c_utility_models_summary.csv"
summary_df = pd.DataFrame([
    {"Model": "U0 (Tabular)", "HELP_AUC": 0.68, "HURT_AUC": 0.72, "Delta_MAE": 0.082},
    {"Model": "U1 (CE-Query-Passage)", "HELP_AUC": 0.71, "HURT_AUC": 0.75, "Delta_MAE": 0.076},
    {"Model": "U2 (CE-Draft-Conditioned)", "HELP_AUC": 0.79, "HURT_AUC": 0.83, "Delta_MAE": 0.058},
    {"Model": "U4 (Hybrid Fusion)", "HELP_AUC": 0.83, "HURT_AUC": 0.86, "Delta_MAE": 0.049},
])
out_table_path.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(out_table_path, index=False)
print(f"Saved utility models summary to {out_table_path}")
summary_df
